# <b>Executive Summary - Task 4: Modelling.</b>

## Task

Build an ML filter for toxic Binance trades that maximizes out-of-sample Score (PnL_kept - PnL_all) under the constraint kept-turnover >= 500k USD/day. Two instruments are considered: BTC and ETH. Three markout horizons are used: 30s, 120s, and 300s. Evaluation is performed using an expanding walk-forward setup W1-W5 with a 7-day embargo.

---

## Methodology

Four models are tested: Rule-based (dist_to_mid), Ridge, LightGBM MSE, and LightGBM Huber. The feature set contains 22 features: microstructure features, Binance and Bybit liquidation features at three time scales, and derived features. The split inside each window is: model_train (80%) -> model training, calibration (20%) -> threshold tuning, test -> one-time OOS evaluation. Statistical comparison is performed using DM-HAC, Hansen SPA, Model Confidence Set, and Romano-Wolf stepdown on daily Scores.

---

## Key Results

### 1. The optimal model depends on the instrument

On BTC, which is a deeper and more liquid market, the relationship between features and PnL is nonlinear. Linear models produce negative Score, worse than no filter. Only LGBM Huber statistically significantly outperforms the baseline: it is the only model in the Model Confidence Set, with Hansen SPA p = 0.005 and Romano-Wolf adjusted p = 0.005.

On ETH, which is a less liquid market, the relationship is more linear at short horizons. Ridge performs well at the 30s horizon. However, at 120s and 300s, the best results are achieved by LGBM Huber.

Practical conclusion: for multi-instrument and multi-horizon deployment, it is optimal to use different models.

### 2. Huber loss is significantly better than MSE on BTC

LightGBM MSE vs LGBM Huber with identical architecture and features: DM = +3.91, p < 0.001. MSE overfits to tail PnL outliers: one trade with PnL = 20 bps affects the gradient 400 times more than a trade with PnL = 1 bps. Huber loss handles outliers more appropriately.

### 3. The optimal operating point is τ = 120s, filter_rate = 0.9

The analysis of daily Score distributions shows that longer horizons and more aggressive filtering produce substantially higher Score:

| Regime | BTC mean Score | ETH mean Score | % days > 0 | Tail risk |
|---|---:|---:|---:|---|
| τ=30s, fr=0.3 (conservative) | +0.12 | +0.14 | 84% | minimal |
| τ=120s, fr=0.9 (main) | +1.57 | +2.10 | 72-78% | controlled |
| τ=300s, fr=0.95 (aggressive) | +3.16 | +3.71 | 73-78% | high, down to -45 bps/day |

### 4. Feature importance structure depends on the horizon

At τ = 30s, dist_to_mid dominates (12%), together with Binance liquidations (10%). At τ = 120s, the importance of Bybit liquidations increases (13%), as well as hour of day (12%). At τ = 300s, Bybit liquidations dominate (20%), while hour is the second most important feature (15%). Short-term toxicity is predicted by the microstructure of the same exchange, while long-term toxicity is driven by cross-exchange liquidation cascades. This supports building a separate signal for each horizon.

On ETH, trade direction (sign) becomes the most important feature at τ = 30s. On a less liquid market, the taker-side signal is stronger.

### 5. The rule-based filter degrades on ETH

The rule-based filter (dist_to_mid) with filter_rate = 0.1-0.2 on ETH systematically filters profitable trades (median pnl_filtered = +0.7 bps). The reason is that dist_to_mid as a single feature cannot distinguish between "price is far because of toxic flow" and "price is far because ETH spreads are wide." The ML model with 22 features learns to distinguish these situations.

### 6. Daily Score behavior

LGBM Huber on BTC (t = 120s, fr = 0.9): mean = +1.57 bps/day, median = +1.42, 78% positive days, std = 2.5. The distribution has a heavy left tail, with 2-3 extremely bad days per quarter.

LGBM Huber on ETH (t = 120s, fr = 0.9): mean = +2.10 bps/day, median = +2.27, 72% positive days, std = 4.5. The distribution is wider than on BTC, as ETH is more volatile.

Ridge on ETH (t = 30s, fr = 0.8): mean = +0.66 bps/day, 81% positive days. The linear model consistently earns on the less liquid instrument.